In [1]:
# ==========================================
# 1. IMPORT LIBRARIES
# ==========================================

import pandas as pd
import numpy as np

from google.colab import files
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score

In [7]:
# ==========================================
# 2. UPLOAD DATASET
# ==========================================

uploaded = files.upload()

file_name = next(iter(uploaded))
df = pd.read_excel(file_name)

print("Dataset uploaded successfully!")
display(df.head())

print("\nColumn names:")
print(df.columns.tolist())

Saving Ridge_Regression_Practice_Dataset.xlsx to Ridge_Regression_Practice_Dataset (3).xlsx
Dataset uploaded successfully!


,Study_Hours,Exam_Marks
0,1.0,32
1,2.0,36
2,3.0,43
3,4.0,47
4,5.0,52



Column names:
['Study_Hours', 'Exam_Marks']


In [8]:


# ==========================================
# 3. SELECT X AND Y
# ==========================================

feature_col = input("Enter input column name (X): ")
target_col = input("Enter target column name (Y): ")

X = df[[feature_col]]
y = df[target_col]

# Convert selected columns to numeric
X = X.apply(pd.to_numeric, errors="coerce")
y = pd.to_numeric(y, errors="coerce")

# Remove missing or invalid rows
data = pd.concat([X, y.rename(target_col)], axis=1).dropna()

X = data[[feature_col]]
y = data[target_col]

print("\nInput feature:", feature_col)
print("Target column:", target_col)


Enter input column name (X): Study_Hours
Enter target column name (Y): Exam_Marks

Input feature: Study_Hours
Target column: Exam_Marks


In [9]:
# ==========================================
# 4. TRAIN-TEST SPLIT
# ==========================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("\nTraining rows:", len(X_train))
print("Testing rows:", len(X_test))


Training rows: 16
Testing rows: 4


In [10]:
# ==========================================
# 5. FEATURE SCALING + RIDGE MODEL
# ==========================================

# StandardScaler scales features.
# Ridge alpha is the regularization strength (lambda).

model = Pipeline([
    ("scaler", StandardScaler()),
    ("ridge", Ridge(alpha=10))
])

In [11]:
# ==========================================
# 6. MODEL FIT
# ==========================================

model.fit(X_train, y_train)

print("\nModel trained successfully!")


Model trained successfully!


In [12]:
# ==========================================
# 7. TEST DATA PREDICTION
# ==========================================

y_pred = model.predict(X_test)

results = pd.DataFrame({
    "Actual Value": y_test.values,
    "Predicted Value": y_pred
})

print("\nTest Predictions:")
display(results)



Test Predictions:


,Actual Value,Predicted Value
0,32,42.223093
1,68,64.072527
2,59,58.246011
3,36,45.136351


In [13]:
# ==========================================
# 8. MODEL EVALUATION
# ==========================================

mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("MSE :", round(mse, 3))
print("RMSE:", round(rmse, 3))
print("R2 Score:", round(r2, 3))


MSE : 50.995
RMSE: 7.141
R2 Score: 0.778


In [21]:
# ==========================================
# 9. CUSTOM DATA PREDICTION
# ==========================================

custom_value = float(
    input(f"Enter {feature_col} value for prediction: ")
)

custom_data = pd.DataFrame({
    feature_col: [custom_value]
})

prediction = model.predict(custom_data)

print("\nCustom Input:", custom_value)
print("Predicted Value:", round(prediction[0], 3))

Enter Study_Hours value for prediction: 5

Custom Input: 5.0
Predicted Value: 53.876


In [15]:

# ==========================================
# SSE + PENALTY + TOTAL LOSS (J)
# ==========================================

# 1. Training data predictions
y_train_pred = model.predict(X_train)

# 2. SSE: Sum of Squared Errors
SSE = np.sum((y_train - y_train_pred) ** 2)

# 3. Get Ridge model and its coefficients
ridge_model = model.named_steps["ridge"]

# 4. Penalty calculation
alpha = ridge_model.alpha
coefficients = ridge_model.coef_

Penalty = alpha * np.sum(coefficients ** 2)

# 5. Total Loss
J = SSE + Penalty

print("SSE      :", round(SSE, 3))
print("Penalty  :", round(Penalty, 3))
print("Total J  :", round(J, 3))


SSE      : 417.408
Penalty  : 660.98
Total J  : 1078.389


In [18]:

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV
import pandas as pd
import numpy as np

# 1. Different alpha values test pannuvom
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("ridge", Ridge())
])

param_grid = {
    "ridge__alpha": [0.001, 0.01, 0.1, 1, 10, 100]
}

# 2. Validation RMSE kammiya irukkura alpha select pannuvom
grid = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    scoring="neg_root_mean_squared_error"
)

grid.fit(X_train, y_train)

best_model = grid.best_estimator_

# 3. Best model-oda SSE, Penalty, J calculate pannuvom
y_pred_train = best_model.predict(X_train)

SSE = np.sum((y_train - y_pred_train) ** 2)

ridge_model = best_model.named_steps["ridge"]
alpha = ridge_model.alpha
coefficients = ridge_model.coef_

Penalty = alpha * np.sum(coefficients ** 2)
J = SSE + Penalty

print("Best Alpha:", alpha)
print("SSE:", round(SSE, 4))
print("Penalty:", round(Penalty, 4))
print("Total J:", round(J, 4))
print("Validation RMSE:", round(-grid.best_score_, 4))

# 4. Ella alpha values-oda comparison
rows = []

for a in param_grid["ridge__alpha"]:
    temp_model = Pipeline([
        ("scaler", StandardScaler()),
        ("ridge", Ridge(alpha=a))
    ])

    temp_model.fit(X_train, y_train)

    pred = temp_model.predict(X_train)
    sse = np.sum((y_train - pred) ** 2)

    coef = temp_model.named_steps["ridge"].coef_
    penalty = a * np.sum(coef ** 2)

    rows.append({
        "Alpha": a,
        "SSE": sse,
        "Penalty": penalty,
        "Total J": sse + penalty
    })

comparison = pd.DataFrame(rows)
display(comparison.round(4))


Best Alpha: 0.1
SSE: 4.4032
Penalty: 17.2379
Total J: 21.641
Validation RMSE: 0.5097


,Alpha,SSE,Penalty,Total J
0,0.001,4.2954,0.1745,4.4700
1,0.010,4.2965,1.7432,6.0397
2,0.100,4.4032,17.2379,21.6410
3,1.000,13.9585,154.6099,168.5685
4,10.000,417.4082,660.9804,1078.3885
5,100.000,2079.6834,332.0621,2411.7455


In [22]:

# ==========================================
# CUSTOM STUDY HOURS PREDICTION
# ==========================================

custom_hours = float(input("Enter Study Hours: "))

custom_data = pd.DataFrame({
    "Study_Hours": [custom_hours]
})

prediction = best_model.predict(custom_data)

print("Study Hours:", custom_hours)
print("Predicted Exam Marks:", round(prediction[0], 2))


Enter Study Hours: 5
Study Hours: 5.0
Predicted Exam Marks: 51.92
